# Preparación de datos V5

Convierte `dataset_limpio.xlsx` en `dataset_preparado.xlsx`. Mantiene un registro diario por fecha, crea variables predictoras comunes y conserva las columnas objetivo de todos los productos.

In [4]:
# ============================================================
# 1. RUTAS Y CONFIGURACIÓN
# ============================================================

BASE = "/content"

INPUT_PATH = f"{BASE}/dataset_limpio.xlsx"
OUTPUT_PATH = f"{BASE}/dataset_preparado.xlsx"
OUTPUT_AUDITORIA = f"{BASE}/auditoria_preparacion_datos.xlsx"

os.makedirs(BASE, exist_ok=True)

PRODUCTOS = ["almuerzo", "sopa", "fanesca", "colada_morada"]

AGG_VENTAS = "sum"

TEMPORADA_FANESCA_MESES = [2, 3]

TEMPORADA_COLADA = {
    "inicio_mes": 10,
    "inicio_dia": 1,
    "fin_mes": 11,
    "fin_dia": 4
}

BASE_CICLO_DIA_SEMANA = 5

In [5]:
# ============================================================
# 2. FUNCIONES AUXILIARES
# ============================================================

def normalize_col(col):
    col = str(col).strip()
    reemplazos = {
        "á": "a", "é": "e", "í": "i", "ó": "o", "ú": "u",
        "Á": "A", "É": "E", "Í": "I", "Ó": "O", "Ú": "U",
        "ñ": "n", "Ñ": "N"
    }
    for a, b in reemplazos.items():
        col = col.replace(a, b)
    col = re.sub(r"[^A-Za-z0-9]+", "_", col)
    col = re.sub(r"_+", "_", col).strip("_")
    return col.lower()

def normalizar_tipo_plato(x):
    if pd.isna(x):
        return np.nan

    s = str(x).strip().lower()
    s = (
        s.replace("á", "a")
        .replace("é", "e")
        .replace("í", "i")
        .replace("ó", "o")
        .replace("ú", "u")
    )
    s = re.sub(r"\s+", "_", s)

    mapping = {
        "menu": "almuerzo",
        "menú": "almuerzo",
        "almuerzo": "almuerzo",
        "sopa": "sopa",
        "fanesca": "fanesca",
        "colada": "colada_morada",
        "coladamorada": "colada_morada",
        "colada_morada": "colada_morada"
    }

    return mapping.get(s, s if s in PRODUCTOS else np.nan)

def es_temporada_fanesca(fecha):
    return int(fecha.month in TEMPORADA_FANESCA_MESES)

def es_temporada_colada(fecha):
    inicio = pd.Timestamp(year=fecha.year, month=TEMPORADA_COLADA["inicio_mes"], day=TEMPORADA_COLADA["inicio_dia"])
    fin = pd.Timestamp(year=fecha.year, month=TEMPORADA_COLADA["fin_mes"], day=TEMPORADA_COLADA["fin_dia"])
    return int(inicio <= fecha <= fin)

def validar_meses_completos(df, columna_fecha="fecha"):
    fecha_min = df[columna_fecha].min()
    fecha_max = df[columna_fecha].max()

    meses_esperados = pd.period_range(
        start=fecha_min.to_period("M"),
        end=fecha_max.to_period("M"),
        freq="M"
    )

    meses_presentes = df[columna_fecha].dt.to_period("M").unique()

    meses_faltantes = [
        str(mes) for mes in meses_esperados
        if mes not in meses_presentes
    ]

    if meses_faltantes:
        raise ValueError(
            "El archivo de entrada está incompleto. "
            f"Faltan estos meses: {', '.join(meses_faltantes)}"
        )

    print("Validación correcta: no faltan meses completos en el rango del dataset.")



In [6]:
# ============================================================
# 3. CARGA Y NORMALIZACIÓN BASE
# ============================================================

df = pd.read_excel(INPUT_PATH)
print("Registros de entrada:", len(df))
print("Columnas originales:", df.columns.tolist())

df.columns = [normalize_col(c) for c in df.columns]

aliases = {
    "fecha": ["fecha"],
    "diasemana": ["diasemana", "dia_semana", "dia_de_la_semana"],
    "tipo_plato": ["tipo_plato", "tipoplato", "tipo_de_plato", "producto", "plato"],
    "clientes": ["clientes", "cantidad_clientes", "cantidad", "platos_vendidos", "unidades"],
    "facturacion": ["facturacion", "facturacion_diaria", "venta", "ventas", "ingreso_diario"],
    "preciomenu": ["preciomenu", "precio_menu", "menu_precio", "precio_almuerzo"],
    "preciosopa": ["preciosopa", "precio_sopa", "sopa_precio"],
    "fanesca_precio": ["fanesca_precio", "precio_fanesca"],
    "coladamorada_precio": ["coladamorada_precio", "precio_coladamorada", "colada_morada_precio", "precio_colada_morada"]
}

resolved = {}
for target, options in aliases.items():
    for opt in options:
        if opt in df.columns:
            resolved[target] = opt
            break

required = ["fecha", "tipo_plato", "clientes"]
faltantes = [c for c in required if c not in resolved]
if faltantes:
    raise ValueError(f"Faltan columnas obligatorias: {faltantes}")

for target, source in resolved.items():
    df[target] = df[source]

for col in ["diasemana", "facturacion", "preciomenu", "preciosopa", "fanesca_precio", "coladamorada_precio"]:
    if col not in df.columns:
        df[col] = np.nan

df = df[[
    "fecha",
    "diasemana",
    "tipo_plato",
    "clientes",
    "facturacion",
    "preciomenu",
    "preciosopa",
    "fanesca_precio",
    "coladamorada_precio"
]].copy()

df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")
df = df.dropna(subset=["fecha"]).copy()

for col in ["clientes", "facturacion", "preciomenu", "preciosopa", "fanesca_precio", "coladamorada_precio"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["clientes"] = df["clientes"].fillna(0)
df = df[df["clientes"] >= 0].copy()

df["tipo_plato"] = df["tipo_plato"].apply(normalizar_tipo_plato)
df = df[df["tipo_plato"].isin(PRODUCTOS)].copy()

df = df.sort_values(["fecha", "tipo_plato"]).reset_index(drop=True)

print("Registros válidos después de normalizar:", len(df))
print("Productos detectados:", sorted(df["tipo_plato"].unique().tolist()))

validar_meses_completos(df)



Registros de entrada: 1501
Columnas originales: ['fecha', 'diasemana', 'tipo_plato', 'clientes', 'facturacion', 'preciomenu', 'preciosopa', 'fanesca_precio', 'coladamorada_precio']
Registros válidos después de normalizar: 1501
Productos detectados: ['almuerzo', 'colada_morada', 'fanesca', 'sopa']
Validación correcta: no faltan meses completos en el rango del dataset.


In [7]:
# ============================================================
# 4. AUDITORÍA DE DUPLICADOS LÓGICOS
# ============================================================

duplicados_logicos = (
    df.groupby(["fecha", "tipo_plato"], as_index=False)
    .size()
    .query("size > 1")
    .sort_values(["fecha", "tipo_plato"])
)

if len(duplicados_logicos) > 0:
    print("Advertencia: existen fechas con más de un registro para el mismo producto.")
    print("Se aplicará la política de agregación:", AGG_VENTAS)
    display(duplicados_logicos.head(20))
else:
    print("No se detectaron duplicados lógicos por fecha y producto.")



Advertencia: existen fechas con más de un registro para el mismo producto.
Se aplicará la política de agregación: sum


,fecha,tipo_plato,size
477,2023-12-13,almuerzo,2
478,2023-12-13,sopa,2
909,2024-10-28,almuerzo,2
1162,2025-05-01,sopa,2
1307,2025-08-19,sopa,2


In [8]:
# ============================================================
# 5. FORMATO ANCHO POR FECHA
# ============================================================

ventas_wide = df.pivot_table(
    index="fecha",
    columns="tipo_plato",
    values="clientes",
    aggfunc=AGG_VENTAS,
    fill_value=0
).reset_index()

for producto in PRODUCTOS:
    if producto not in ventas_wide.columns:
        ventas_wide[producto] = 0

precios = df.groupby("fecha", as_index=False)[[
    "preciomenu",
    "preciosopa",
    "fanesca_precio",
    "coladamorada_precio"
]].max()

df_model = ventas_wide.merge(precios, on="fecha", how="left")
df_model = df_model.sort_values("fecha").reset_index(drop=True)



In [9]:
# ============================================================
# 6. VARIABLES PREDICTORAS
# ============================================================

fecha_min_modelo = df_model["fecha"].min()

df_model["anio"] = df_model["fecha"].dt.year
df_model["mes"] = df_model["fecha"].dt.month
df_model["dia_mes"] = df_model["fecha"].dt.day
df_model["dia_semana_num"] = df_model["fecha"].dt.weekday
df_model["semana_anio"] = df_model["fecha"].dt.isocalendar().week.astype(int)

df_model["es_fanesca_temporada"] = df_model["fecha"].apply(es_temporada_fanesca).astype(int)
df_model["es_colada_temporada"] = df_model["fecha"].apply(es_temporada_colada).astype(int)

df_model["es_inicio_mes"] = (df_model["dia_mes"] <= 5).astype(int)
df_model["es_quincena"] = df_model["dia_mes"].between(13, 17).astype(int)
df_model["es_fin_mes"] = (df_model["dia_mes"] >= 25).astype(int)

df_model["es_lunes"] = (df_model["dia_semana_num"] == 0).astype(int)
df_model["es_martes"] = (df_model["dia_semana_num"] == 1).astype(int)
df_model["es_miercoles"] = (df_model["dia_semana_num"] == 2).astype(int)
df_model["es_jueves"] = (df_model["dia_semana_num"] == 3).astype(int)
df_model["es_viernes"] = (df_model["dia_semana_num"] == 4).astype(int)

df_model["mes_sin"] = np.sin(2 * np.pi * df_model["mes"] / 12)
df_model["mes_cos"] = np.cos(2 * np.pi * df_model["mes"] / 12)

df_model["dia_semana_sin"] = np.sin(2 * np.pi * df_model["dia_semana_num"] / BASE_CICLO_DIA_SEMANA)
df_model["dia_semana_cos"] = np.cos(2 * np.pi * df_model["dia_semana_num"] / BASE_CICLO_DIA_SEMANA)

df_model["tendencia"] = (df_model["fecha"] - fecha_min_modelo).dt.days
df_model["tendencia_log"] = np.log1p(df_model["tendencia"])
df_model["crecimiento_anual"] = df_model["anio"] - fecha_min_modelo.year



In [10]:
# ============================================================
# 7. CONTROL ESTACIONAL DEL TARGET
# ============================================================

df_model.loc[df_model["es_fanesca_temporada"] == 0, "fanesca"] = 0
df_model.loc[df_model["es_colada_temporada"] == 0, "colada_morada"] = 0

# ============================================================
# 8. TRATAMIENTO DE PRECIOS
# ============================================================

for col in ["preciomenu", "preciosopa", "fanesca_precio", "coladamorada_precio"]:
    df_model[col] = df_model[col].ffill().bfill().fillna(0)



In [11]:
# ============================================================
# 9. VARIABLES FINALES
# ============================================================

variables_objetivo = PRODUCTOS

variables_predictoras = [
    "anio",
    "mes",
    "dia_mes",
    "dia_semana_num",
    "semana_anio",

    "es_fanesca_temporada",
    "es_colada_temporada",
    "es_inicio_mes",
    "es_quincena",
    "es_fin_mes",

    "es_lunes",
    "es_martes",
    "es_miercoles",
    "es_jueves",
    "es_viernes",

    "mes_sin",
    "mes_cos",
    "dia_semana_sin",
    "dia_semana_cos",

    "tendencia",
    "tendencia_log",
    "crecimiento_anual",

    "preciomenu",
    "preciosopa",
    "fanesca_precio",
    "coladamorada_precio"
]

dataset_preparado = df_model[["fecha"] + variables_predictoras + variables_objetivo].copy()

faltantes_finales = [
    col for col in ["fecha"] + variables_predictoras + variables_objetivo
    if col not in dataset_preparado.columns
]
if faltantes_finales:
    raise ValueError(f"Faltan columnas finales: {faltantes_finales}")

if dataset_preparado[variables_predictoras].isna().any().any():
    columnas_na = dataset_preparado[variables_predictoras].columns[
        dataset_preparado[variables_predictoras].isna().any()
    ].tolist()
    raise ValueError(f"Existen nulos en variables predictoras: {columnas_na}")



In [13]:
# ============================================================
# 10. EXPORTAR ARCHIVOS EXCEL DE FORMA SEGURA
# ============================================================

import os
from pathlib import Path
from google.colab import files

# Asegurar que las rutas sean tipo Path
OUTPUT_PATH = Path(OUTPUT_PATH)
OUTPUT_AUDITORIA = Path(OUTPUT_AUDITORIA)

# Crear carpeta de salida si no existe
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_AUDITORIA.parent.mkdir(parents=True, exist_ok=True)

# Eliminar archivos anteriores para evitar conflictos o archivos corruptos
if OUTPUT_PATH.exists():
    OUTPUT_PATH.unlink()

if OUTPUT_AUDITORIA.exists():
    OUTPUT_AUDITORIA.unlink()

# Crear copia segura del dataset preparado
dataset_export = dataset_preparado.copy()

# Asegurar formato correcto de fecha
dataset_export["fecha"] = pd.to_datetime(dataset_export["fecha"], errors="coerce")

# Ordenar columnas y registros
dataset_export = dataset_export.sort_values("fecha").reset_index(drop=True)

# Validar que no existan fechas nulas antes de exportar
if dataset_export["fecha"].isna().any():
    raise ValueError("Existen fechas inválidas en dataset_preparado. Corrige antes de exportar.")

# Validar que no existan nulos en columnas predictoras
columnas_con_nulos = dataset_export.columns[dataset_export.isna().any()].tolist()

if columnas_con_nulos:
    print("Advertencia: existen columnas con valores nulos:")
    print(columnas_con_nulos)

# ============================================================
# 10.1 EXPORTAR DATASET_PREPARADO
# ============================================================

with pd.ExcelWriter(
    OUTPUT_PATH,
    engine="openpyxl",
    datetime_format="YYYY-MM-DD"
) as writer:
    dataset_export.to_excel(
        writer,
        sheet_name="dataset_preparado",
        index=False
    )

print("Dataset preparado exportado correctamente:")
print(OUTPUT_PATH)


# ============================================================
# 10.2 CREAR RESUMEN POR PRODUCTO
# ============================================================

resumen_productos = []

for producto in PRODUCTOS:
    resumen_productos.append({
        "producto": producto,
        "total_clientes": int(dataset_export[producto].sum()),
        "dias_con_demanda": int((dataset_export[producto] > 0).sum()),
        "promedio_diario": round(float(dataset_export[producto].mean()), 2),
        "maximo_diario": int(dataset_export[producto].max()),
        "minimo_diario": int(dataset_export[producto].min())
    })

df_resumen_productos = pd.DataFrame(resumen_productos)


# ============================================================
# 10.3 CREAR DESCRIPCIÓN DEL DATASET
# ============================================================

df_describe = dataset_export.describe(include="all").reset_index()

# Convertir columnas tipo fecha a texto en describe para evitar conflictos de Excel
for col in df_describe.columns:
    if pd.api.types.is_datetime64_any_dtype(df_describe[col]):
        df_describe[col] = df_describe[col].astype(str)


# ============================================================
# 10.4 EXPORTAR ARCHIVO DE AUDITORÍA
# ============================================================

with pd.ExcelWriter(
    OUTPUT_AUDITORIA,
    engine="openpyxl",
    datetime_format="YYYY-MM-DD"
) as writer:
    df_describe.to_excel(
        writer,
        sheet_name="describe",
        index=False
    )

    df_resumen_productos.to_excel(
        writer,
        sheet_name="resumen_productos",
        index=False
    )

    if "duplicados_logicos" in globals():
        duplicados_logicos.to_excel(
            writer,
            sheet_name="duplicados_logicos",
            index=False
        )

print("Archivo de auditoría exportado correctamente:")
print(OUTPUT_AUDITORIA)


# ============================================================
# 10.5 VALIDAR QUE LOS EXCEL SE PUEDAN LEER
# ============================================================

try:
    prueba_dataset = pd.read_excel(OUTPUT_PATH)
    prueba_auditoria = pd.read_excel(OUTPUT_AUDITORIA, sheet_name="resumen_productos")

    print("Validación correcta: los archivos Excel se pueden leer nuevamente.")
    print("Filas leídas en dataset_preparado:", len(prueba_dataset))
    print("Filas leídas en resumen_productos:", len(prueba_auditoria))

except Exception as e:
    raise ValueError(f"El archivo Excel fue generado, pero no se pudo leer correctamente: {e}")


# ============================================================
# 10.6 MOSTRAR RESULTADOS
# ============================================================

print("Preparación finalizada correctamente.")
print("Dataset preparado:", OUTPUT_PATH)
print("Auditoría:", OUTPUT_AUDITORIA)
print("Filas finales:", len(dataset_export))
print("Rango:", dataset_export["fecha"].min().date(), "a", dataset_export["fecha"].max().date())

display(dataset_export.head())
display(df_resumen_productos)


# ============================================================
# 10.7 DESCARGAR ARCHIVOS DESDE COLAB
# ============================================================

files.download(str(OUTPUT_PATH))
files.download(str(OUTPUT_AUDITORIA))

Dataset preparado exportado correctamente:
/content/dataset_preparado.xlsx
Archivo de auditoría exportado correctamente:
/content/auditoria_preparacion_datos.xlsx
Validación correcta: los archivos Excel se pueden leer nuevamente.
Filas leídas en dataset_preparado: 740
Filas leídas en resumen_productos: 4
Preparación finalizada correctamente.
Dataset preparado: /content/dataset_preparado.xlsx
Auditoría: /content/auditoria_preparacion_datos.xlsx
Filas finales: 740
Rango: 2023-01-02 a 2025-12-31


,fecha,anio,mes,dia_mes,dia_semana_num,semana_anio,es_fanesca_temporada,es_colada_temporada,es_inicio_mes,es_quincena,...,tendencia_log,crecimiento_anual,preciomenu,preciosopa,fanesca_precio,coladamorada_precio,almuerzo,sopa,fanesca,colada_morada
0,2023-01-02,2023,1,2,0,1,0,0,1,0,...,0.000000,0,4.0,1.5,7.0,2.0,104,72,0,0
1,2023-01-03,2023,1,3,1,1,0,0,1,0,...,0.693147,0,4.0,1.5,7.0,2.0,106,54,0,0
2,2023-01-04,2023,1,4,2,1,0,0,1,0,...,1.098612,0,4.0,1.5,7.0,2.0,97,72,0,0
3,2023-01-05,2023,1,5,3,1,0,0,1,0,...,1.386294,0,4.0,1.5,7.0,2.0,103,0,0,0
4,2023-01-06,2023,1,6,4,1,0,0,0,0,...,1.609438,0,4.0,1.5,7.0,2.0,123,0,0,0


,producto,total_clientes,dias_con_demanda,promedio_diario,maximo_diario,minimo_diario
0,almuerzo,75225,737,101.66,231,0
1,sopa,44370,661,59.96,192,0
2,fanesca,652,35,0.88,54,0
3,colada_morada,1567,63,2.12,46,0


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>